In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures,LabelEncoder,StandardScaler
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.metrics import r2_score,root_mean_squared_error,mean_squared_error,mean_absolute_error
import pickle as pk

In [69]:
df_old = pd.read_csv("laptop_data.csv")

In [70]:
df_old.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [71]:
df_old['Cpu'].unique()

array(['Intel Core i5 2.3GHz', 'Intel Core i5 1.8GHz',
       'Intel Core i5 7200U 2.5GHz', 'Intel Core i7 2.7GHz',
       'Intel Core i5 3.1GHz', 'AMD A9-Series 9420 3GHz',
       'Intel Core i7 2.2GHz', 'Intel Core i7 8550U 1.8GHz',
       'Intel Core i5 8250U 1.6GHz', 'Intel Core i3 6006U 2GHz',
       'Intel Core i7 2.8GHz', 'Intel Core M m3 1.2GHz',
       'Intel Core i7 7500U 2.7GHz', 'Intel Core i7 2.9GHz',
       'Intel Core i3 7100U 2.4GHz', 'Intel Atom x5-Z8350 1.44GHz',
       'Intel Core i5 7300HQ 2.5GHz', 'AMD E-Series E2-9000e 1.5GHz',
       'Intel Core i5 1.6GHz', 'Intel Core i7 8650U 1.9GHz',
       'Intel Atom x5-Z8300 1.44GHz', 'AMD E-Series E2-6110 1.5GHz',
       'AMD A6-Series 9220 2.5GHz',
       'Intel Celeron Dual Core N3350 1.1GHz',
       'Intel Core i3 7130U 2.7GHz', 'Intel Core i7 7700HQ 2.8GHz',
       'Intel Core i5 2.0GHz', 'AMD Ryzen 1700 3GHz',
       'Intel Pentium Quad Core N4200 1.1GHz',
       'Intel Atom x5-Z8550 1.44GHz',
       'Intel Celeron Du

In [72]:
df_old['Cpu'].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz       190
Intel Core i7 7700HQ 2.8GHz      146
Intel Core i7 7500U 2.7GHz       134
Intel Core i7 8550U 1.8GHz        73
Intel Core i5 8250U 1.6GHz        72
                                ... 
Intel Core M M3-6Y30 0.9GHz        1
AMD A9-Series 9420 2.9GHz          1
Intel Core i3 6006U 2.2GHz         1
AMD A6-Series 7310 2GHz            1
Intel Xeon E3-1535M v6 3.1GHz      1
Name: count, Length: 118, dtype: int64

In [73]:
df_old["Preprocessor"]=df_old['Cpu'].str.split(' ').str.slice(0,3).str.join(' ')

In [74]:
df_old["Preprocessor"].value_counts()

Preprocessor
Intel Core i7               527
Intel Core i5               423
Intel Core i3               136
Intel Celeron Dual           80
Intel Pentium Quad           27
Intel Core M                 19
AMD A9-Series 9420           12
Intel Celeron Quad            8
AMD A6-Series 9220            8
AMD A12-Series 9720P          7
Intel Atom x5-Z8350           5
AMD A8-Series 7410            4
Intel Atom x5-Z8550           4
Intel Pentium Dual            3
AMD A9-Series 9410            3
AMD Ryzen 1700                3
AMD A9-Series A9-9420         2
AMD A10-Series 9620P          2
Intel Atom X5-Z8350           2
AMD E-Series E2-9000e         2
Intel Xeon E3-1535M           2
Intel Xeon E3-1505M           2
AMD E-Series 7110             2
AMD A10-Series 9600P          2
AMD A6-Series A6-9220         2
AMD A10-Series A10-9620P      2
AMD Ryzen 1600                1
Intel Atom x5-Z8300           1
AMD E-Series E2-6110          1
AMD FX 9830P                  1
AMD E-Series E2-9000       

In [75]:
df_demo = df_old["Preprocessor"].value_counts()
df_demo

Preprocessor
Intel Core i7               527
Intel Core i5               423
Intel Core i3               136
Intel Celeron Dual           80
Intel Pentium Quad           27
Intel Core M                 19
AMD A9-Series 9420           12
Intel Celeron Quad            8
AMD A6-Series 9220            8
AMD A12-Series 9720P          7
Intel Atom x5-Z8350           5
AMD A8-Series 7410            4
Intel Atom x5-Z8550           4
Intel Pentium Dual            3
AMD A9-Series 9410            3
AMD Ryzen 1700                3
AMD A9-Series A9-9420         2
AMD A10-Series 9620P          2
Intel Atom X5-Z8350           2
AMD E-Series E2-9000e         2
Intel Xeon E3-1535M           2
Intel Xeon E3-1505M           2
AMD E-Series 7110             2
AMD A10-Series 9600P          2
AMD A6-Series A6-9220         2
AMD A10-Series A10-9620P      2
AMD Ryzen 1600                1
Intel Atom x5-Z8300           1
AMD E-Series E2-6110          1
AMD FX 9830P                  1
AMD E-Series E2-9000       

In [76]:
 # if df_demo < 100:
 #     df_old["Preprocessor"] = df_old['Preprocessor'].replace("Others")


In [77]:
df_old["Preprocessor"].unique()

array(['Intel Core i5', 'Intel Core i7', 'AMD A9-Series 9420',
       'Intel Core i3', 'Intel Core M', 'Intel Atom x5-Z8350',
       'AMD E-Series E2-9000e', 'Intel Atom x5-Z8300',
       'AMD E-Series E2-6110', 'AMD A6-Series 9220', 'Intel Celeron Dual',
       'AMD Ryzen 1700', 'Intel Pentium Quad', 'Intel Atom x5-Z8550',
       'AMD FX 9830P', 'AMD E-Series 6110', 'Intel Xeon E3-1505M',
       'AMD E-Series 9000e', 'AMD A10-Series A10-9620P',
       'AMD A6-Series A6-9220', 'AMD A10-Series 9600P',
       'AMD A8-Series 7410', 'AMD A12-Series 9720P', 'Intel Celeron Quad',
       'AMD Ryzen 1600', 'AMD A10-Series 9620P', 'AMD E-Series 7110',
       'AMD A9-Series A9-9420', 'Intel Xeon E3-1535M',
       'AMD E-Series E2-9000', 'AMD A6-Series 7310', 'Intel Atom Z8350',
       'Intel Pentium Dual', 'AMD A12-Series 9700P', 'AMD A4-Series 7210',
       'AMD FX 8800P', 'Intel Atom X5-Z8350', 'Samsung Cortex A72&A53',
       'AMD E-Series 9000', 'AMD A9-Series 9410'], dtype=object)

In [78]:
# df_li = ['Intel Core i5', 'Intel Core i7', 'Intel Core i3']
# for i in df_old["Preprocessor"].items:
#      if i != df_li:
#       i = i.replace("Others")
    

In [79]:
df_old["Preprocessor"] = df_old["Preprocessor"].replace(['AMD A9-Series 9420', 'Intel Core M', 'Intel Atom x5-Z8350','AMD E-Series E2-9000e', 'Intel Atom x5-Z8300','AMD E-Series E2-6110', 'AMD A6-Series 9220', 'Intel Celeron Dual','AMD Ryzen 1700', 'Intel Pentium Quad', 'Intel Atom x5-Z8550','AMD FX 9830P', 'AMD E-Series 6110', 'Intel Xeon E3-1505M','AMD E-Series 9000e', 'AMD A10-Series A10-9620P','AMD A6-Series A6-9220', 'AMD A10-Series 9600P','AMD A8-Series 7410', 'AMD A12-Series 9720P', 'Intel Celeron Quad','AMD Ryzen 1600', 'AMD A10-Series 9620P', 'AMD E-Series 7110','AMD A9-Series A9-9420', 'Intel Xeon E3-1535M','AMD E-Series E2-9000', 'AMD A6-Series 7310', 'Intel Atom Z8350','Intel Pentium Dual', 'AMD A12-Series 9700P', 'AMD A4-Series 7210','AMD FX 8800P', 'Intel Atom X5-Z8350', 'Samsung Cortex A72&A53','AMD E-Series 9000', 'AMD A9-Series 9410'],"Others")

In [80]:
df_old["Preprocessor"]

0       Intel Core i5
1       Intel Core i5
2       Intel Core i5
3       Intel Core i7
4       Intel Core i5
            ...      
1298    Intel Core i7
1299    Intel Core i7
1300           Others
1301    Intel Core i7
1302           Others
Name: Preprocessor, Length: 1303, dtype: object

In [81]:
df_old["Preprocessor"].value_counts()

Preprocessor
Intel Core i7    527
Intel Core i5    423
Others           217
Intel Core i3    136
Name: count, dtype: int64

In [82]:
df_old.drop(columns="Cpu",inplace=True)

In [83]:
df_old["Weight"] = df_old['Weight'].str.replace("kg","").astype("float")

In [84]:
df_old.rename(columns={"Weight":"Weight_kg"},inplace=True)

In [85]:
df_old.select_dtypes(include=["int","float"])

,Unnamed: 0,Inches,Weight_kg,Price
0,0,13.3,1.37,71378.6832
1,1,13.3,1.34,47895.5232
2,2,15.6,1.86,30636.0000
3,3,15.4,1.83,135195.3360
4,4,13.3,1.37,96095.8080
...,...,...,...,...
1298,1298,14.0,1.80,33992.6400
1299,1299,13.3,1.30,79866.7200
1300,1300,14.0,1.50,12201.1200
1301,1301,15.6,2.19,40705.9200


In [86]:
df_old[["Screen_width","Screen_height"]] = df_old["ScreenResolution"].str.split().str[-1].str.split('x',expand=True).astype("int")

In [87]:
df_old["Full HD"] = df_old["ScreenResolution"].str.contains("Full HD")
df_old["IPS"] = df_old["ScreenResolution"].str.contains("IPS")
df_old["Touchscreen"] = df_old["ScreenResolution"].str.contains("Touchscreen")
df_old["Quad HD+"] = df_old["ScreenResolution"].str.contains("Quad HD+")
df_old["4K Ultra HD"] = df_old["ScreenResolution"].str.contains("4K Ultra HD")
df_old.drop(columns="ScreenResolution",inplace=True)

In [88]:
df_old

,Unnamed: 0,Company,TypeName,Inches,Ram,Memory,Gpu,OpSys,Weight_kg,Price,Preprocessor,Screen_width,Screen_height,Full HD,IPS,Touchscreen,Quad HD+,4K Ultra HD
0,0,Apple,Ultrabook,13.3,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832,Intel Core i5,2560,1600,False,True,False,False,False
1,1,Apple,Ultrabook,13.3,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232,Intel Core i5,1440,900,False,False,False,False,False
2,2,HP,Notebook,15.6,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000,Intel Core i5,1920,1080,True,False,False,False,False
3,3,Apple,Ultrabook,15.4,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360,Intel Core i7,2880,1800,False,True,False,False,False
4,4,Apple,Ultrabook,13.3,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080,Intel Core i5,2560,1600,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1298,Lenovo,2 in 1 Convertible,14.0,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.80,33992.6400,Intel Core i7,1920,1080,True,True,True,False,False
1299,1299,Lenovo,2 in 1 Convertible,13.3,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.30,79866.7200,Intel Core i7,3200,1800,False,True,True,True,False
1300,1300,Lenovo,Notebook,14.0,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.50,12201.1200,Others,1366,768,False,False,False,False,False
1301,1301,HP,Notebook,15.6,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19,40705.9200,Intel Core i7,1366,768,False,False,False,False,False


In [89]:
df_old['Ram'].unique()
df_old.rename(columns={"Ram":"Ram_GB"},inplace=True)
df_old['Ram_GB'] = df_old['Ram_GB'].str.replace('GB','').astype("int")

In [90]:
split_col = df_old["Gpu"].str.split()
df_old["Gpu_Brand"] = split_col.str[0]
df_old.drop(columns="Gpu",inplace=True)

In [91]:
df_old

,Unnamed: 0,Company,TypeName,Inches,Ram_GB,Memory,OpSys,Weight_kg,Price,Preprocessor,Screen_width,Screen_height,Full HD,IPS,Touchscreen,Quad HD+,4K Ultra HD,Gpu_Brand
0,0,Apple,Ultrabook,13.3,8,128GB SSD,macOS,1.37,71378.6832,Intel Core i5,2560,1600,False,True,False,False,False,Intel
1,1,Apple,Ultrabook,13.3,8,128GB Flash Storage,macOS,1.34,47895.5232,Intel Core i5,1440,900,False,False,False,False,False,Intel
2,2,HP,Notebook,15.6,8,256GB SSD,No OS,1.86,30636.0000,Intel Core i5,1920,1080,True,False,False,False,False,Intel
3,3,Apple,Ultrabook,15.4,16,512GB SSD,macOS,1.83,135195.3360,Intel Core i7,2880,1800,False,True,False,False,False,AMD
4,4,Apple,Ultrabook,13.3,8,256GB SSD,macOS,1.37,96095.8080,Intel Core i5,2560,1600,False,True,False,False,False,Intel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1298,Lenovo,2 in 1 Convertible,14.0,4,128GB SSD,Windows 10,1.80,33992.6400,Intel Core i7,1920,1080,True,True,True,False,False,Intel
1299,1299,Lenovo,2 in 1 Convertible,13.3,16,512GB SSD,Windows 10,1.30,79866.7200,Intel Core i7,3200,1800,False,True,True,True,False,Intel
1300,1300,Lenovo,Notebook,14.0,2,64GB Flash Storage,Windows 10,1.50,12201.1200,Others,1366,768,False,False,False,False,False,Intel
1301,1301,HP,Notebook,15.6,6,1TB HDD,Windows 10,2.19,40705.9200,Intel Core i7,1366,768,False,False,False,False,False,AMD


In [92]:
def process_memory(text):
    ssd = hdd = flash = hybrid = 0

    parts = text.split('+')

    for part in parts:
        part = part.strip()
        words = part.split()

        size_unit = words[0] 

        if 'TB' in size_unit:
            size = float(size_unit.replace('TB', '')) * 1024
        else:
            size = float(size_unit.replace('GB', ''))

        if 'SSD' in part:
            ssd += size
        elif 'HDD' in part:
            hdd += size
        elif 'Flash' in part:
            flash += size
        elif 'Hybrid' in part:
            hybrid += size

    return pd.Series([ssd, hdd, flash, hybrid])

In [93]:
df_old[["SSD","HDD","Flash_Storage","Hybrid"]] =  df_old['Memory'].apply(process_memory)

In [94]:
df_old

,Unnamed: 0,Company,TypeName,Inches,Ram_GB,Memory,OpSys,Weight_kg,Price,Preprocessor,...,Full HD,IPS,Touchscreen,Quad HD+,4K Ultra HD,Gpu_Brand,SSD,HDD,Flash_Storage,Hybrid
0,0,Apple,Ultrabook,13.3,8,128GB SSD,macOS,1.37,71378.6832,Intel Core i5,...,False,True,False,False,False,Intel,128.0,0.0,0.0,0.0
1,1,Apple,Ultrabook,13.3,8,128GB Flash Storage,macOS,1.34,47895.5232,Intel Core i5,...,False,False,False,False,False,Intel,0.0,0.0,128.0,0.0
2,2,HP,Notebook,15.6,8,256GB SSD,No OS,1.86,30636.0000,Intel Core i5,...,True,False,False,False,False,Intel,256.0,0.0,0.0,0.0
3,3,Apple,Ultrabook,15.4,16,512GB SSD,macOS,1.83,135195.3360,Intel Core i7,...,False,True,False,False,False,AMD,512.0,0.0,0.0,0.0
4,4,Apple,Ultrabook,13.3,8,256GB SSD,macOS,1.37,96095.8080,Intel Core i5,...,False,True,False,False,False,Intel,256.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1298,Lenovo,2 in 1 Convertible,14.0,4,128GB SSD,Windows 10,1.80,33992.6400,Intel Core i7,...,True,True,True,False,False,Intel,128.0,0.0,0.0,0.0
1299,1299,Lenovo,2 in 1 Convertible,13.3,16,512GB SSD,Windows 10,1.30,79866.7200,Intel Core i7,...,False,True,True,True,False,Intel,512.0,0.0,0.0,0.0
1300,1300,Lenovo,Notebook,14.0,2,64GB Flash Storage,Windows 10,1.50,12201.1200,Others,...,False,False,False,False,False,Intel,0.0,0.0,64.0,0.0
1301,1301,HP,Notebook,15.6,6,1TB HDD,Windows 10,2.19,40705.9200,Intel Core i7,...,False,False,False,False,False,AMD,0.0,1024.0,0.0,0.0


In [95]:
df_old.drop(columns=["Unnamed: 0","Memory"],inplace=True)

In [96]:
df_old

,Company,TypeName,Inches,Ram_GB,OpSys,Weight_kg,Price,Preprocessor,Screen_width,Screen_height,Full HD,IPS,Touchscreen,Quad HD+,4K Ultra HD,Gpu_Brand,SSD,HDD,Flash_Storage,Hybrid
0,Apple,Ultrabook,13.3,8,macOS,1.37,71378.6832,Intel Core i5,2560,1600,False,True,False,False,False,Intel,128.0,0.0,0.0,0.0
1,Apple,Ultrabook,13.3,8,macOS,1.34,47895.5232,Intel Core i5,1440,900,False,False,False,False,False,Intel,0.0,0.0,128.0,0.0
2,HP,Notebook,15.6,8,No OS,1.86,30636.0000,Intel Core i5,1920,1080,True,False,False,False,False,Intel,256.0,0.0,0.0,0.0
3,Apple,Ultrabook,15.4,16,macOS,1.83,135195.3360,Intel Core i7,2880,1800,False,True,False,False,False,AMD,512.0,0.0,0.0,0.0
4,Apple,Ultrabook,13.3,8,macOS,1.37,96095.8080,Intel Core i5,2560,1600,False,True,False,False,False,Intel,256.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,Lenovo,2 in 1 Convertible,14.0,4,Windows 10,1.80,33992.6400,Intel Core i7,1920,1080,True,True,True,False,False,Intel,128.0,0.0,0.0,0.0
1299,Lenovo,2 in 1 Convertible,13.3,16,Windows 10,1.30,79866.7200,Intel Core i7,3200,1800,False,True,True,True,False,Intel,512.0,0.0,0.0,0.0
1300,Lenovo,Notebook,14.0,2,Windows 10,1.50,12201.1200,Others,1366,768,False,False,False,False,False,Intel,0.0,0.0,64.0,0.0
1301,HP,Notebook,15.6,6,Windows 10,2.19,40705.9200,Intel Core i7,1366,768,False,False,False,False,False,AMD,0.0,1024.0,0.0,0.0


In [97]:
df_old['Company'].value_counts()

Company
Dell         297
Lenovo       297
HP           274
Asus         158
Acer         103
MSI           54
Toshiba       48
Apple         21
Samsung        9
Razer          7
Mediacom       7
Microsoft      6
Xiaomi         4
Vero           4
Chuwi          3
Google         3
Fujitsu        3
LG             3
Huawei         2
Name: count, dtype: int64

In [98]:
df_old['TypeName'].value_counts()

TypeName
Notebook              727
Gaming                205
Ultrabook             196
2 in 1 Convertible    121
Workstation            29
Netbook                25
Name: count, dtype: int64

In [99]:
df_old['Inches'].unique()

array([13.3, 15.6, 15.4, 14. , 12. , 11.6, 17.3, 10.1, 13.5, 12.5, 13. ,
       18.4, 13.9, 12.3, 17. , 15. , 14.1, 11.3])

In [100]:
df_old['Ram_GB'].unique()

array([ 8, 16,  4,  2, 12,  6, 32, 24, 64])

In [101]:
df_old['OpSys'].value_counts()

OpSys
Windows 10      1072
No OS             66
Linux             62
Windows 7         45
Chrome OS         27
macOS             13
Mac OS X           8
Windows 10 S       8
Android            2
Name: count, dtype: int64

In [102]:
df_old['Weight_kg'].unique()

array([1.37 , 1.34 , 1.86 , 1.83 , 2.1  , 2.04 , 1.3  , 1.6  , 2.2  ,
       0.92 , 1.22 , 0.98 , 2.5  , 1.62 , 1.91 , 2.3  , 1.35 , 1.88 ,
       1.89 , 1.65 , 2.71 , 1.2  , 1.44 , 2.8  , 2.   , 2.65 , 2.77 ,
       3.2  , 0.69 , 1.49 , 2.4  , 2.13 , 2.43 , 1.7  , 1.4  , 1.8  ,
       1.9  , 3.   , 1.252, 2.7  , 2.02 , 1.63 , 1.96 , 1.21 , 2.45 ,
       1.25 , 1.5  , 2.62 , 1.38 , 1.58 , 1.85 , 1.23 , 1.26 , 2.16 ,
       2.36 , 2.05 , 1.32 , 1.75 , 0.97 , 2.9  , 2.56 , 1.48 , 1.74 ,
       1.1  , 1.56 , 2.03 , 1.05 , 4.4  , 1.29 , 1.95 , 2.06 , 1.12 ,
       1.42 , 3.49 , 3.35 , 2.23 , 4.42 , 2.69 , 2.37 , 4.7  , 3.6  ,
       2.08 , 4.3  , 1.68 , 1.41 , 4.14 , 2.18 , 2.24 , 2.67 , 2.14 ,
       1.36 , 2.25 , 2.15 , 2.19 , 2.54 , 3.42 , 1.28 , 2.33 , 1.45 ,
       2.79 , 1.84 , 2.6  , 2.26 , 3.25 , 1.59 , 1.13 , 1.78 , 1.15 ,
       1.27 , 1.43 , 2.31 , 1.16 , 1.64 , 2.17 , 1.47 , 3.78 , 1.79 ,
       0.91 , 1.99 , 4.33 , 1.93 , 1.87 , 2.63 , 3.4  , 3.14 , 1.94 ,
       1.24 , 4.6  ,

In [103]:
df_old['Preprocessor'].unique()

array(['Intel Core i5', 'Intel Core i7', 'Others', 'Intel Core i3'],
      dtype=object)

In [104]:
df_old['Gpu_Brand'].unique()

array(['Intel', 'AMD', 'Nvidia', 'ARM'], dtype=object)

In [105]:
df_old['SSD'].unique()

array([ 128.,    0.,  256.,  512.,   32.,   64., 1024.,   16.,  768.,
        180.,  240.,    8.])

In [106]:
df_old['Screen_height'].unique()

array([1600,  900, 1080, 1800,  768, 1440, 1200, 1504, 2160, 1824])

In [107]:
df_old['Screen_width'].unique()

array([2560, 1440, 1920, 2880, 1366, 2304, 3200, 2256, 3840, 2160, 1600,
       2736, 2400])

In [108]:
df_old.select_dtypes(include=["object","boolean"]).keys()

Index(['Company', 'TypeName', 'OpSys', 'Preprocessor', 'Full HD', 'IPS',
       'Touchscreen', 'Quad HD+', '4K Ultra HD', 'Gpu_Brand'],
      dtype='object')

In [109]:
df_old.rename(columns={"Full HD": "Full_HD"},inplace=True)
df_old.rename(columns={"Quad HD+": "Quad_HD"},inplace=True)
df_old.rename(columns={"4K Ultra HD": "Ultra_HD"},inplace=True)


In [110]:
df_old['Full_HD'] = df_old['Full_HD'].replace("True","1").replace("False","0").astype(int)
df_old['IPS'] = df_old['IPS'].replace("True","1").replace("False","0").astype(int)
df_old['Touchscreen'] = df_old['Touchscreen'].replace("True","1").replace("False","0").astype(int)
df_old['Quad_HD'] = df_old['Quad_HD'].replace("True","1").replace("False","0").astype(int)
df_old['Ultra_HD'] = df_old['Ultra_HD'].replace("True","1").replace("False","0").astype(int)
# IPS	Touchscreen	Quad HD+	4K Ultra HD

In [ ]:
df_old['Company'] = LabelEncoder().fit_transform(df_old['Company'])
df_old['TypeName'] = LabelEncoder().fit_transform(df_old['TypeName'])
df_old['OpSys'] = LabelEncoder().fit_transform(df_old['OpSys'])
df_old['Preprocessor'] = LabelEncoder().fit_transform(df_old['Preprocessor'])
df_old['Gpu_Brand'] = LabelEncoder().fit_transform(df_old['Gpu_Brand'])

In [112]:
df_old

,Company,TypeName,Inches,Ram_GB,OpSys,Weight_kg,Price,Preprocessor,Screen_width,Screen_height,Full_HD,IPS,Touchscreen,Quad_HD,Ultra_HD,Gpu_Brand,SSD,HDD,Flash_Storage,Hybrid
0,1,4,13.3,8,8,1.37,71378.6832,1,2560,1600,0,1,0,0,0,2,128.0,0.0,0.0,0.0
1,1,4,13.3,8,8,1.34,47895.5232,1,1440,900,0,0,0,0,0,2,0.0,0.0,128.0,0.0
2,7,3,15.6,8,4,1.86,30636.0000,1,1920,1080,1,0,0,0,0,2,256.0,0.0,0.0,0.0
3,1,4,15.4,16,8,1.83,135195.3360,2,2880,1800,0,1,0,0,0,0,512.0,0.0,0.0,0.0
4,1,4,13.3,8,8,1.37,96095.8080,1,2560,1600,0,1,0,0,0,2,256.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,10,0,14.0,4,5,1.80,33992.6400,2,1920,1080,1,1,1,0,0,2,128.0,0.0,0.0,0.0
1299,10,0,13.3,16,5,1.30,79866.7200,2,3200,1800,0,1,1,1,0,2,512.0,0.0,0.0,0.0
1300,10,3,14.0,2,5,1.50,12201.1200,3,1366,768,0,0,0,0,0,2,0.0,0.0,64.0,0.0
1301,7,3,15.6,6,5,2.19,40705.9200,2,1366,768,0,0,0,0,0,0,0.0,1024.0,0.0,0.0


In [113]:
x = df_old.drop(columns="Price")
y = df_old[["Price"]]

In [114]:
x.keys()

Index(['Company', 'TypeName', 'Inches', 'Ram_GB', 'OpSys', 'Weight_kg',
       'Preprocessor', 'Screen_width', 'Screen_height', 'Full_HD', 'IPS',
       'Touchscreen', 'Quad_HD', 'Ultra_HD', 'Gpu_Brand', 'SSD', 'HDD',
       'Flash_Storage', 'Hybrid'],
      dtype='object')

In [115]:
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=23,test_size=0.2)

In [116]:
scaler = StandardScaler()

In [117]:
scaled_x_train = scaler.fit_transform(x_train)
scaled_x_test = scaler.transform(x_test)

# Linear regression

In [118]:
model1 = LinearRegression()

In [119]:
model1.fit(scaled_x_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [120]:
y_pred = model1.predict(scaled_x_test)

In [121]:
r2_score(y_test,y_pred)

0.723137286680873

# Ridge and Lasso

In [122]:
# re = Ridge(alpha=2)

In [123]:
# re.fit(x_train,y_train)

In [124]:
# re.score(x_test,y_test)

In [125]:
# la = Lasso(alpha=2)

In [126]:
# la.fit(x_train,y_train)

In [127]:
# la.score(x_test,y_test)

In [128]:
pk.dump(scaler,open('scaler.pkl','wb'))
pk.dump(model1,open('model1.pkl','wb'))
pk.dump(encoders, open("encoders.pkl", "wb"))


In [129]:
# pk.dump(label,open('label.pkl','wb'))